# Treinamento do modelo final (Google Colab)

Ambiente: **Ambiente de execução → Alterar tipo → GPU T4**. Rode as células na ordem.

Entrada: um `.zip` com as imagens `img_XXXX.jpg` (1024×768) e os rótulos `img_XXXX.txt` no formato YOLO.
Saída: `modelo_final.zip` com `modelo_final.onnx`, `best.pt`, `classes.txt`, `results.csv`, `args.yaml` e os gráficos do treino.

In [ ]:
!pip -q install ultralytics==8.4.214
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 1. Enviar o zip das fotos
Pelo navegador (arquivos pequenos) **ou** pelo Google Drive (recomendado para arquivos grandes). Use só uma das duas células.

In [ ]:
# opcao A: enviar pelo navegador
from google.colab import files
files.upload()

In [ ]:
# opcao B: copiar do Google Drive (coloque o zip na raiz do Meu Drive)
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/fotos_treino_leve.zip" /content/

## 2. Separar treino e validação
Blocos de 10 imagens consecutivas: um bloco a cada dez vai para a validação (imagens vizinhas são quase iguais e não podem cair em conjuntos diferentes).

In [ ]:
import zipfile, glob, os, re, shutil
zips = glob.glob('/content/*.zip')
zp = max(zips, key=os.path.getmtime)
print('usando:', zp)
shutil.rmtree('/content/fotos', ignore_errors=True)
zipfile.ZipFile(zp).extractall('/content/fotos')
imgs = glob.glob('/content/fotos/**/img_*.jpg', recursive=True)
base = os.path.dirname(imgs[0])
nums = sorted(int(re.findall(r'\d+', os.path.basename(f))[0]) for f in imgs)
nums = [n for n in nums if os.path.exists(f'{base}/img_{n:04d}.txt')]
print('fotos com rotulo:', len(nums))
for s in ('train', 'val'):
    for t in ('images', 'labels'):
        shutil.rmtree(f'/content/ds/{t}/{s}', ignore_errors=True); os.makedirs(f'/content/ds/{t}/{s}')
nv = 0
for n in nums:
    s = 'val' if (n // 10) % 10 == 0 else 'train'
    nv += s == 'val'
    shutil.copy(f'{base}/img_{n:04d}.jpg', f'/content/ds/images/{s}/')
    shutil.copy(f'{base}/img_{n:04d}.txt', f'/content/ds/labels/{s}/')
print('treino:', len(nums) - nv, '| validacao:', nv)
nomes = [l.strip() for l in '''gancho_pequeno
afastador_hohmann_8x236
afastador_hohmann_mini_2x8
afastador_mini_2x16
retrator_senn_mueller
pinca_reducao_dentes_14cm
pinca_reducao_dentes_16cm
pinca_verbrugge
pinca_reducao_pontas_152mm
retrator_weitlaner
pinca_reducao_pontas_20cm
modelador_pequeno
descolador_williger
retrator_farabeuf
instrumento_desconhecido'''.splitlines()]
with open('/content/data_final.yaml', 'w') as f:
    f.write('path: /content/ds\ntrain: images/train\nval: images/val\nnames:\n')
    for i, nm in enumerate(nomes): f.write(f'  {i}: {nm}\n')
open('/content/classes.txt', 'w').write('\n'.join(nomes) + '\n')

## 3. Treinar, exportar e baixar
O resultado também é copiado para o Google Drive (se ele foi montado na opção B).

In [ ]:
from ultralytics import YOLO
import shutil, os, glob
m = YOLO('yolo26s.pt')
m.train(data='/content/data_final.yaml', epochs=100, imgsz=1024, batch=8,
        project='/content/runs', name='final', patience=20, seed=0,
        cache=True, workers=2, flipud=0.5, save_period=10)
run = str(m.trainer.save_dir)
YOLO(f'{run}/weights/best.pt').export(format='onnx', imgsz=1024)
out = '/content/modelo_final'
os.makedirs(out, exist_ok=True)
shutil.copy(f'{run}/weights/best.onnx', f'{out}/modelo_final.onnx')
for f in ['weights/best.pt', 'results.csv', 'args.yaml']:
    shutil.copy(f'{run}/{f}', out)
shutil.copy('/content/classes.txt', out)
for f in glob.glob(f'{run}/*.png') + glob.glob(f'{run}/*.jpg'):
    shutil.copy(f, out)
zipf = shutil.make_archive('/content/modelo_final', 'zip', out)
if os.path.isdir('/content/drive/MyDrive'):
    shutil.copy(zipf, '/content/drive/MyDrive/modelo_final.zip')
from google.colab import files
files.download(zipf)